In [3]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [4]:

import pandas as pd
#df=pd.read_csv("...Data/data.csv")
df = pd.read_csv("Data/data.csv")

In [5]:

agg_features = df.groupby('CustomerId').agg({
    'Amount': [
        'sum',    # Total Transaction Amount
        'mean',   # Average Transaction Amount
        'count',  # Transaction Count
        'std'     # Standard Deviation (Variability)
    ]
}).reset_index()

# 3. Rename the columns for clarity (flattening the multi-index)
agg_features.columns = [
    'CustomerId', 
    'Total_Transaction_Amount', 
    'Average_Transaction_Amount', 
    'Transaction_Count', 
    'Std_Dev_Transaction_Amount'
]

# 4. Handle NaN in Standard Deviation
# If a customer only has 1 transaction, the Std Dev will be NaN. 
# It is best practice to fill this with 0.
agg_features['Std_Dev_Transaction_Amount'] = agg_features['Std_Dev_Transaction_Amount'].fillna(0)

print(agg_features.head())

        CustomerId  Total_Transaction_Amount  Average_Transaction_Amount  \
0     CustomerId_1                  -10000.0               -10000.000000   
1    CustomerId_10                  -10000.0               -10000.000000   
2  CustomerId_1001                   20000.0                 4000.000000   
3  CustomerId_1002                    4225.0                  384.090909   
4  CustomerId_1003                   20000.0                 3333.333333   

   Transaction_Count  Std_Dev_Transaction_Amount  
0                  1                    0.000000  
1                  1                    0.000000  
2                  5                 6558.963333  
3                 11                  560.498966  
4                  6                 6030.478146  


In [6]:
# Ensure the column is in datetime format
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# Extracting components
df['Transaction_Hour'] = df['TransactionStartTime'].dt.hour
df['Transaction_Day'] = df['TransactionStartTime'].dt.day
df['Transaction_Month'] = df['TransactionStartTime'].dt.month
df['Transaction_Year'] = df['TransactionStartTime'].dt.year

print(df[['TransactionStartTime', 'Transaction_Hour', 'Transaction_Day']].head())

       TransactionStartTime  Transaction_Hour  Transaction_Day
0 2018-11-15 02:18:49+00:00                 2               15
1 2018-11-15 02:19:08+00:00                 2               15
2 2018-11-15 02:44:21+00:00                 2               15
3 2018-11-15 03:32:55+00:00                 3               15
4 2018-11-15 03:34:21+00:00                 3               15


Defining feature groups

In [7]:
# 1. Define Column Groups
# Use the aggregate and datetime features you just created
num_cols = [
    'Total_Transaction_Amount', 'Average_Transaction_Amount', 
    'Transaction_Count', 'Std_Dev_Transaction_Amount', 
    'Transaction_Hour', 'Transaction_Day' , 'Amount' , 'Value' , 'TransactionStartTime' , 'PricingStrategy' , 'FraudResult'
]
cat_cols = [  'BatchId' , 'AccountId' , 'SubscriptionId' , 'CustomerId' , 'CurrencyCode'  , 'ProviderId' , 'ProductId' , 'ProductCategory' , 'ChannelId']

In [8]:
# 2. Build the "Robust & Automated" Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        # Handle Missing Values (Median) + Standardization
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        
        # Handle Missing Values (Constant) + One-Hot Encoding
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])

In [9]:
clf = Pipeline(steps=[('preprocessor', preprocessor)])